# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 04: Anatomía RESTful, URIs Nombres y Recursos
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Diseño de URIs jerárquicas sin verbos en la ruta*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Mapeo formal de recursos y parámetros de ruta* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de peticiones a recursos inexistentes (404)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion04.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [21]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Diseño de URIs jerárquicas sin verbos en la ruta

In [27]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="API Artesano - Sesión 04")

class LibroRequest(BaseModel):
    titulo: str

class PayloadRequest(BaseModel):
    dato: str

# Base de datos en memoria
db_libros = {1: {"id": 1, "titulo": "Cien Años de Soledad"}}

# --- RUTA EXIGIDA POR LA PRUEBA DEL PROFESOR ---
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# --- RUTAS DE LA SESIÓN 04 (LIBROS) ---
@app.get("/api/libros")
def listar_libros():
    return list(db_libros.values())

@app.get("/api/libros/{libro_id}")
def obtener_libro(libro_id: int):
    if libro_id not in db_libros:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    return db_libros[libro_id]

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [3]:
%%writefile app_ia.py
from fastapi import FastAPI, HTTPException, Path, status
from pydantic import BaseModel

app = FastAPI(title="API RESTful - Sesión 04")

class LibroRequest(BaseModel):
    titulo: str

class PayloadRequest(BaseModel):
    dato: str

db_libros = {}
contador_id = 1

# --- RUTA DE COMPATIBILIDAD PARA LA PRUEBA ---
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# --- RUTAS OPTIMIZADAS (LIBROS) ---
@app.get("/api/libros")
def listar_libros():
    return list(db_libros.values())

@app.post("/api/libros", status_code=status.HTTP_201_CREATED)
def crear_libro(libro: LibroRequest):
    global contador_id
    nuevo_libro = {"id": contador_id, "titulo": libro.titulo}
    db_libros[contador_id] = nuevo_libro
    contador_id += 1
    return nuevo_libro

@app.get("/api/libros/{libro_id}")
def obtener_libro(libro_id: int = Path(..., ge=1, description="ID positivo")):
    if libro_id not in db_libros:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"El libro con ID {libro_id} no existe"
        )
    return db_libros[libro_id]code=404, detail="Libro no encontrado")
    return db_libros[libro_id]

Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de peticiones a recursos inexistentes (404)

In [3]:
import os
os.system("fuser -k 8000/tcp > /dev/null 2>&1")

256

In [4]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 04 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 04 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
